# CLASS 6
# Machine Learning Package - Neural Networks (Training)

In [10]:
import numpy as np

from si.data.dataset import Dataset
from si.metrics.nn_accuracy import nn_accuracy
from si.neural_networks.activation import SigmoidActivation
from si.neural_networks.layers import DenseLayer
from si.neural_networks.losses import MeanSquaredError
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.optimizers import SGD

## Exercise — Train the XOR with the NeuralNetwork

In [11]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype = float)
y = np.array([0, 1, 1, 0], dtype = float)
ds = Dataset(X, y)

np.random.seed(1)

net = NeuralNetwork(epochs        = 5000, 
                    batch_size    = 4, 
                    optimizer     = SGD, 
                    learning_rate = 0.5,
                    loss          = MeanSquaredError, 
                    metric        = nn_accuracy)

net.add(DenseLayer(4, (2,)))
net.add(SigmoidActivation())
net.add(DenseLayer(1))
net.add(SigmoidActivation())

net.fit(ds)

print(net.predict(ds).round(2))     
print('Accuracy:', net.score(ds))
print('Final loss:', round(net.history[5000]['loss'], 4))

[[0.05]
 [0.95]
 [0.95]
 [0.07]]
Accuracy: 1.0
Final loss: 0.003


### Variations — What do you observe?

In [12]:
def train_xor(seed = 1, learning_rate = 0.5, middle_activation = True):

    np.random.seed(seed)

    net = NeuralNetwork(epochs        = 5000, 
                        batch_size    = 4, 
                        optimizer     = SGD, 
                        learning_rate = learning_rate,
                        loss          = MeanSquaredError, 
                        metric        = nn_accuracy)

    net.add(DenseLayer(4, (2,)))
    if middle_activation:
        net.add(SigmoidActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())

    net.fit(ds)

    print(f'predictions: {net.predict(ds).round(2).ravel()}   '
          f'accuracy: {net.score(ds):.2f}   '
          f'final loss: {net.history[5000]["loss"]:.4f}')


print('Original:                  ', end = ' ')
train_xor()

print('Without middle Sigmoid:    ', end = ' ')
train_xor(middle_activation = False)

print('learning_rate = 0.01:      ', end = ' ')
train_xor(learning_rate = 0.01)

print('np.random.seed(0):         ', end = ' ')
train_xor(seed = 0)

Original:                   predictions: [0.05 0.95 0.95 0.07]   accuracy: 1.00   final loss: 0.0030
Without middle Sigmoid:     predictions: [0.5 0.5 0.5 0.5]   accuracy: 0.50   final loss: 0.2500
learning_rate = 0.01:       predictions: [0.5 0.5 0.5 0.5]   accuracy: 0.50   final loss: 0.2499
np.random.seed(0):          predictions: [0.5 0.5 0.5 0.5]   accuracy: 0.50   final loss: 0.2500


**Observations:**

- **Without the middle Sigmoid:** two Dense layers in a row are equivalent to a single linear transformation, so the network is just a logistic regression. The XOR is not linearly separable, so it cannot be learned: the network predicts 0.5 for every sample (accuracy 0.5). The non-linear activation is what allows the hidden layer to learn the XOR.
- **learning_rate = 0.01:** the steps of gradient descent are too small. After 5000 epochs the weights have barely moved from the initial values, the loss is still ~0.25 and the predictions stay at 0.5. The network would need many more epochs to converge.
- **np.random.seed(0):** with other initial weights, the same network does not learn the XOR and gets stuck at 0.5 (a plateau/local minimum of the loss). Training a neural network depends on the weight initialization.

## Exercise 14 — CategoricalCrossEntropy
**Loss on a small one-hot example and derivative check (finite differences).**

In [13]:
from si.neural_networks.losses import CategoricalCrossEntropy

cce = CategoricalCrossEntropy()

y_true = np.array([[1, 0, 0],
                   [0, 1, 0],
                   [0, 0, 1]], dtype = float)

y_pred = np.array([[0.7, 0.2, 0.1],
                   [0.1, 0.8, 0.1],
                   [0.3, 0.3, 0.4]])

print('Loss:', cce.loss(y_true, y_pred))     # -(log 0.7 + log 0.8 + log 0.4)

num, eps = np.zeros_like(y_pred), 1e-6

for i in range(3):
    for j in range(3):
        p = y_pred.copy(); p[i, j] += eps
        m = y_pred.copy(); m[i, j] -= eps
        num[i, j] = (cce.loss(y_true, p) - cce.loss(y_true, m)) / (2 * eps)

print('Derivative check:', np.abs(num - cce.derivative(y_true, y_pred)).max())   # ~1e-10

Loss: 1.4961092271270973
Derivative check: 8.580314236894537e-11


## Exercise 15 — Adam optimizer
**Test 1: minimize L(w) = (w − 3)², starting at w = 0.**

In [14]:
from si.neural_networks.optimizers import Adam

adam = Adam(learning_rate = 0.1)
w    = np.array([0.])

for step in range(1, 201):
    w = adam.update(w, 2 * (w - 3))          # dL/dw = 2(w - 3)
    if step in (1, 2, 3, 50, 100, 200):
        print(f'step {step:>3}: w = {w[0]:.4f}')

step   1: w = 0.1000
step   2: w = 0.1999
step   3: w = 0.2996
step  50: w = 3.1689
step 100: w = 2.9807
step 200: w = 3.0001


**Test 2: train the XOR with Adam (same network as before).**

In [15]:
np.random.seed(1)

net_adam = NeuralNetwork(epochs        = 2000, 
                         batch_size    = 4, 
                         optimizer     = Adam, 
                         learning_rate = 0.05,
                         loss          = MeanSquaredError, 
                         metric        = nn_accuracy)

net_adam.add(DenseLayer(4, (2,)))
net_adam.add(SigmoidActivation())
net_adam.add(DenseLayer(1))
net_adam.add(SigmoidActivation())

net_adam.fit(ds)

print(net_adam.predict(ds).round(2))
print('Accuracy:', net_adam.score(ds))

[[0.  ]
 [0.99]
 [1.  ]
 [0.01]]
Accuracy: 1.0


## Exercise 16 — Build, train and evaluate a neural network
Binary classification dataset with 32 features (generated: the label depends on a linear combination of the features, so it can be learned).

In [16]:
from si.model_selection.cross_validate import k_fold_cross_validation
from si.neural_networks.activation import ReLUActivation
from si.neural_networks.losses import BinaryCrossEntropy

np.random.seed(42)

n_samples, n_features = 500, 32

X_32   = np.random.randn(n_samples, n_features)
true_w = np.random.randn(n_features)
y_32   = (X_32 @ true_w > 0).astype(float)

ds_32 = Dataset(X_32, y_32)

print('Shape:', ds_32.shape())
print('Class counts:', np.bincount(y_32.astype(int)))

Shape: (500, 32)
Class counts: [231 269]


**Build the network: Dense (32 → 16) → ReLU → Dense (16 → 8) → ReLU → Dense (8 → 1) → Sigmoid.**

In [17]:
net_16 = NeuralNetwork(epochs        = 100, 
                       batch_size    = 16, 
                       optimizer     = SGD, 
                       learning_rate = 0.01,
                       loss          = BinaryCrossEntropy, 
                       metric        = nn_accuracy)

net_16.add(DenseLayer(n_features // 2, (n_features,)))   # 32 -> 16
net_16.add(ReLUActivation())
net_16.add(DenseLayer(n_features // 4))                  # 16 -> 8
net_16.add(ReLUActivation())
net_16.add(DenseLayer(1))                                # output
net_16.add(SigmoidActivation())

for layer in net_16.layers:
    print(f'{layer.layer_name():<18} output shape: {layer.output_shape()}   parameters: {layer.parameters()}')

DenseLayer         output shape: (16,)   parameters: 528
ReLUActivation     output shape: (16,)   parameters: 0
DenseLayer         output shape: (8,)   parameters: 136
ReLUActivation     output shape: (8,)   parameters: 0
DenseLayer         output shape: (1,)   parameters: 9
SigmoidActivation  output shape: (1,)   parameters: 0


**Evaluate with 5-fold cross validation.**

In [18]:
scores = k_fold_cross_validation(net_16, ds_32, cv = 5, seed = 42)

print('Accuracy per fold:', np.round(scores, 4))
print(f'Mean accuracy: {np.mean(scores):.4f} ± {np.std(scores):.4f}')

Accuracy per fold: [0.92 0.91 0.94 0.89 0.92]
Mean accuracy: 0.9160 ± 0.0162
